In [32]:
import torch
import torch.nn as nn
import torch.optim as optim

pairs = [
    ("hello", "hi how are you"),
    ("how are you", "i am fine what about you"),
    ("what is your name", "my name is chatbot"),
    ("bye", "goodbye take care")
]

word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
idx2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}

for inp, out in pairs:
    for word in (inp + " " + out).split():
        if word not in word2idx:
            idx = len(word2idx)
            word2idx[word] = idx
            idx2word[idx] = word

vocab_size = len(word2idx)

def encode(sentence, add_tokens=False):
    tokens = [word2idx.get(w, 3) for w in sentence.split()]
    if add_tokens:
        tokens = [1] + tokens + [2]
    return tokens


# ======================
# BEFORE ATTENTION
# ======================
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb=32, hidden=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb)
        self.encoder = nn.LSTM(emb, hidden, batch_first=True)
        self.decoder = nn.LSTM(emb, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, src, tgt):
        _, (hidden, cell) = self.encoder(self.embedding(src))
        out, _ = self.decoder(self.embedding(tgt), (hidden, cell))
        return self.fc(out)


# ======================
# AFTER ATTENTION
# ======================
class AttentionSeq2Seq(nn.Module):
    def __init__(self, vocab_size, emb=32, hidden=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb)
        self.encoder = nn.LSTM(emb, hidden, batch_first=True)
        self.decoder = nn.LSTM(emb, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, src, tgt):
        enc_out, (hidden, cell) = self.encoder(self.embedding(src))
        dec_out, _ = self.decoder(self.embedding(tgt), (hidden, cell))

        scores = torch.bmm(dec_out, enc_out.transpose(1, 2))
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)

        out = dec_out + context
        return self.fc(out)


def train_model(model, epochs):
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for _ in range(epochs):
        for inp, out in pairs:
            src = torch.tensor([encode(inp)], dtype=torch.long)
            tgt = torch.tensor([encode(out, True)], dtype=torch.long)

            optimizer.zero_grad()
            pred = model(src, tgt[:, :-1])

            loss = criterion(
                pred.reshape(-1, vocab_size),
                tgt[:, 1:].reshape(-1)
            )
            loss.backward()
            optimizer.step()


def generate(model, sentence, max_len=10):
    src = torch.tensor([encode(sentence)], dtype=torch.long)
    generated = [word2idx["<sos>"]]

    for _ in range(max_len):
        tgt = torch.tensor([generated], dtype=torch.long)
        pred = model(src, tgt)
        next_word = torch.argmax(pred[0, -1]).item()

        if next_word == word2idx["<eos>"]:
            break
        generated.append(next_word)

    return " ".join(idx2word[i] for i in generated[1:])


# ======================
# TRAIN DIFFERENTLY
# ======================
before_model = Seq2Seq(vocab_size)
after_model = AttentionSeq2Seq(vocab_size)

train_model(before_model, epochs=1)
train_model(after_model, epochs=1000)

# ======================
# TEST
# ======================
test = "how are you"

print("===== BEFORE ATTENTION =====")
print("Input:", test)
print("Output:", generate(before_model, test))

print("\n===== AFTER ATTENTION =====")
print("Input:", test)
print("Output:", generate(after_model, test))

===== BEFORE ATTENTION =====
Input: how are you
Output: i am am fine what

===== AFTER ATTENTION =====
Input: how are you
Output: i am fine what about you
